# 02 Pedal characterization

Analyzes a session captured with `uv run pedalrig pedal`. Every capture is divided by the cable
loopback (gain G, latency, polarity, and the same-stimulus reference) from the calibration session
recorded in `meta.json` → `calibration_ref`. Sections skip themselves when their captures are absent.

In [ ]:
%matplotlib inline
%load_ext autoreload
# pick up edits to pedalrig without restarting the kernel
%autoreload 2
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

if Path.cwd().name == "notebooks":  # sessions/ lives in the project root
    os.chdir("..")

from pedalrig import analysis as A
from pedalrig import plots as P
from pedalrig.session import Session, latest_calibration, list_sessions

In [ ]:
ps_path = list_sessions("pedal")[-1]     # or: ps_path = Path("sessions/<dir>")
ps = Session(ps_path)
cs = Session(ps.meta["calibration_ref"] or latest_calibration())
cal = A.calibration_summary(cs)
pa = A.PedalAnalysis(ps, cal, cs)
have = set(ps.capture_names())
print("pedal session:", ps_path)
print("calibration:  ", cs.path)
print(f"G = {cal['G_db']:.3f} dB, latency {cal['latency_samples']} samples, polarity {cal['polarity']:+d}, "
      f"harmonics valid to {pa.f_valid_hi:.0f} Hz")
for c in ps.meta["captures"]:
    flags = " ".join(k for k in ("clipped", "xrun", "kept_with_warning") if c[k])
    print(f"  {c['name']:<26} peak {c['peak_dbfs'][0]:6.1f} dBFS  {flags}")


LEVELS = {"m50": "small-signal -50 dBFS", "m30": "quiet -30 dBFS", "m10": "loud -10 dBFS"}


def knob_sweeps(test, knob, tag):
    pos = ("min", "noon", "max")
    lvs = list(LEVELS)
    if not all(f"{test}_sweep_{lv}_{tag}-{p}" in have for lv in lvs for p in pos):
        print(f"no complete {test} captures in this session")
        return None
    res = {p: {lv: pa.sweep_response(f"{test}_sweep_{lv}_{tag}-{p}") for lv in lvs} for p in pos}
    for lv in lvs:
        P.plot_fr([(f"{knob}={p}", r[lv]["f"], r[lv]["mag_db"], r[lv]["phase_deg"]) for p, r in res.items()],
                  title=f"{test}: response at {LEVELS[lv]} (re loopback)")
    P.plot_hd([(f"{knob}={p}", r["m10"]["hd"]) for p, r in res.items()],
              floor=res["noon"]["m10"]["ref_hd"], title=f"{test}: distortion at -10 dBFS")
    for p, r in res.items():
        print(f"{knob}={p:<5} mean THD 200-2000 Hz: "
              + ", ".join(f"{LEVELS[lv]} {A.band_mean(r[lv]['f'], r[lv]['hd']['THD'], 200, 2000):6.1f} dB" for lv in lvs)
              + "  (measured after the pedal's filters: a post-clip low-pass also lowers it)")
    # A linear filter after the clipper changes the response by the same dB at every level;
    # one before/inside the clipper is partly flattened by clipping at the loud level.
    # Compare the clean -50 dBFS response with the loud one.
    lo = "m50"
    F = np.array([100, 200, 500, 1000, 2000, 3000, 5000, 10000])
    for p in ("min", "max"):
        f = res[p][lo]["f"]
        dq = np.interp(F, f, res[p][lo]["mag_db"] - res["noon"][lo]["mag_db"])
        dl = np.interp(F, f, res[p]["m10"]["mag_db"] - res["noon"]["m10"]["mag_db"])
        print(f"{knob}={p} re noon at {F.tolist()} Hz:\n   {LEVELS[lo]:<22} {np.round(dq, 1)}\n"
              f"   {LEVELS['m10']:<22} {np.round(dl, 1)}\n   max |difference| {np.max(np.abs(dl - dq)):.1f} dB")
    print("same change at both levels (<= 1 dB) = linear filter AFTER the clipper; "
          "level-dependent = before or inside it")
    return res


def fmt_onset(r):
    """1 dB compression onset re the clean gain at the ramp's quiet start. If the pedal is already
    compressing there (slope < 0.95), the true onset is lower: report it as an upper bound."""
    if r["onset_1db_dbu"] is None:
        return "none"
    if not r["start_linear"]:
        return f"<= {r['onset_1db_dbu']:.1f} dBu (not linear even at {r['in_dbu'][0]:.1f} dBu: slope {r['start_slope']:.2f})"
    return f"{r['onset_1db_dbu']:.1f} dBu"


## Basic response — all knobs at noon, three input levels

The pedal's frequency response (re the loopback) with every knob at noon, at −50 dBFS (−35.5 dBu, small-signal),
−30 dBFS (−15.5 dBu) and −10 dBFS (+4.5 dBu). The −50 dBFS curve is the pedal's EQ before clipping; the drop of the
louder curves at each frequency is how much that frequency is compressed at that level. Taken from the T2 Tone=noon
captures (all knobs noon).

In [ ]:
BASIC = {lv: f"T2_sweep_{lv}_tone-noon" for lv in LEVELS}
if all(n in have for n in BASIC.values()):
    basic = {lv: pa.sweep_response(n) for lv, n in BASIC.items()}
    _ = P.plot_fr([(LEVELS[lv], r["f"], r["mag_db"], r["phase_deg"]) for lv, r in basic.items()],
                  title="Pedal response, all knobs noon (re loopback)")
    _ = P.plot_hd([(LEVELS[lv], r["hd"]) for lv, r in basic.items()],
                  floor=basic["m10"]["ref_hd"], title="Distortion vs frequency, all knobs noon")
    ss = basic["m50"]
    i_pk = int(np.argmax(ss["mag_db"]))
    print(f"small-signal peak {ss['mag_db'][i_pk]:+.1f} dB at {ss['f'][i_pk]:.0f} Hz")
    F = [20, 50, 100, 200, 500, 1000, 2000, 3000, 5000, 10000, 18000]
    print(f"{'Hz':>7} " + " ".join(f"{f:>7}" for f in F))
    for lv, r in basic.items():
        print(f"{lv:>7} " + " ".join(f"{np.interp(f, r['f'], r['mag_db']):+7.1f}" for f in F) + "  dB")
    for lv in ("m30", "m10"):
        comp = [np.interp(f, ss["f"], ss["mag_db"]) - np.interp(f, basic[lv]["f"], basic[lv]["mag_db"])
                for f in F]  # response is gain, so the gain lost re small-signal = compression
        print(f"compression at {LEVELS[lv]:<14} " + " ".join(f"{c:7.1f}" for c in comp) + "  dB")
else:
    print("no complete T2 Tone=noon captures in this session")


## Q1 / Q2 — Where does it clip, how hard, and what does Gain change? (T1)

1. **Where does it clip, and how hard?** 200 Hz sine ramp −40 → −6 dBFS over 10 s, all knobs at noon. Plot output sample vs. input sample. Straight = clean; the bend = onset; gentle bend = soft clipping; sharp knee = hard clipping; top/bottom halves differ = asymmetric (more even harmonics). Repeat at three Gain positions.
2. **What does Gain change?** Same ramp at Gain min/noon/max. Only the onset moves → pre-clip boost. Curve shape changes → multiple stages or diode switching. Note how output level changes with Gain.

In [ ]:
gain = [n for n in ("T1_ramp_200_gain-min", "T1_ramp_200_gain-noon", "T1_ramp_200_gain-max") if n in have]
if gain:
    t1 = []
    for n in gain:
        r = pa.ramp_transfer(n)
        r["label"] = "Gain=" + n.rsplit("-", 1)[1]
        t1.append(r)
        print(f"{r['label']:<10} 1 dB onset: {fmt_onset(r)}   clean gain (ramp start) {r['linear_gain_db']:+.1f} dB"
              f"   inverted={r['inverted']}")
    _ = P.plot_amp_transfer(t1, "T1 200 Hz ramp vs Gain")
    _ = P.plot_scatter_transfer(t1)
    _ = P.plot_asym(t1)
else:
    print("no T1 Gain captures")

In [ ]:
freq = [n for n in ("T1_ramp_100", "T1_ramp_200_gain-noon", "T1_ramp_400", "T1_ramp_1k", "T1_ramp_3k") if n in have]
if len(freq) > 1:
    tf = []
    for n in freq:
        r = pa.ramp_transfer(n)
        r["label"] = f"{r['f']:.0f} Hz"
        tf.append(r)
        print(f"{r['label']:<8} 1 dB onset: {fmt_onset(r)}   clean gain (ramp start) {r['linear_gain_db']:+.1f} dB")
    _ = P.plot_amp_transfer(tf, "T1 ramp vs frequency (Gain noon)")

## Q3 — Is Tone before or after the clipper? (T2)

3. **Is Tone before or after the clipper?** Quiet sweep (−30 dBFS) at Tone min/noon/max gives the filter shapes. Loud sweep (−10 dBFS): if Tone changes *how* it distorts, it is pre-clip; if it only re-EQs the same grit, it is post-clip. Pre-clip tone controls affect feel and pick response.

In [ ]:
t2 = knob_sweeps("T2", "Tone", "tone")

## Q4 — What is Saturation-Cut? (T3)

4. **What is Saturation-Cut?** Same test. Expected: a post-clip low-pass whose corner moves with the knob and no change to the clipping curve. If it alters the clipping curve or the bass entering the clipper, it is something else.

In [ ]:
t3 = knob_sweeps("T3", "Cut", "cut")

## Q5 — Any dynamics? (T4)

5. **Any dynamics?** Bursts of 200 Hz at −10 dBFS. Flat envelope = memoryless. Sag/overshoot = compression or supply sag. Notes that start clean then dirty (or vice versa) = bias shift.

In [ ]:
if "T4_bursts_200_m10" in have:
    b = pa.burst_dynamics("T4_bursts_200_m10")
    print("sag per burst (dB, last full cycle - 2nd cycle):", np.round(b["cap"]["sag_db"], 2))
    print("loopback ref sag:                                ", np.round(b["ref"]["sag_db"], 2))
    _ = P.plot_bursts(b)
else:
    print("no T4 capture")

## Q6 — Clean blend? (T2, Tone noon)

6. **Clean blend?** Compare low end of loud vs. quiet sweeps. Clean-blend designs keep bass tight; straight clippers distort everything equally.

In [ ]:
if {"T2_sweep_m50_tone-noon", "T2_sweep_m10_tone-noon"} <= have:
    bl = pa.blend_metric("T2_sweep_m50_tone-noon", "T2_sweep_m10_tone-noon")
    qlab = "small-signal -50"
    _ = P.plot_fr([(qlab, bl["f"], bl["quiet_db"], None), ("loud -10", bl["f"], bl["loud_db"], None)],
                  title="Q6: response normalized at 1 kHz")
    print(f"mean THD at -10 dBFS: 40-150 Hz {bl['thd_low_40_150_db']:.1f} dB, "
          f"400-2000 Hz {bl['thd_mid_400_2k_db']:.1f} dB (bass much cleaner = clean blend)")
else:
    print("no T2 Tone-noon captures")

## Volume check (T5)

Verify Volume once: at three settings with a quiet sweep, confirm only the level shifts, then fix it and ignore it.

In [ ]:
vol = [n for n in ("T5_sweep_m30_vol-9", "T5_sweep_m30_vol-noon", "T5_sweep_m30_vol-3") if n in have]
if "T5_sweep_m30_vol-noon" in vol:
    vc = pa.volume_check(vol)
    for n, m in vc["metrics"].items():
        print(f"{n:<24} level re noon {m['level_re_noon_db']:+6.2f} dB   shape deviation {m['shape_dev_db']:.2f} dB")
    ok = all(m["shape_dev_db"] <= 0.5 for m in vc["metrics"].values())
    print("Volume only shifts level" if ok else "Volume also changes the frequency response")
    _ = P.plot_fr([(n, r["f"], r["mag_db"], None) for n, r in vc["responses"].items()], title="T5 Volume")
else:
    print("no T5 captures")

## Reading the results for chain placement

- **Clipping onset in dBu** says how hot a signal it wants. Low onset → saturates on everything; place after a volume control or compressor. High onset → wants a boost in front.
- **Pre-clip Tone** → EQ and boosts *before* the pedal matter a lot. **Post-clip Tone** → EQ after it does the same job.
- **Post-clip low-pass (Cut)** does cab-like work; with a cab sim or amp after it, Cut can likely stay open.
- **Output level** at preferred settings determines how it stacks into the next pedal's input.